# 저비트 양자화(PTQ) 실습 — FP4 · MXFP4 · KL Clipping · AdaRound · Hadamard

저비트(4-bit 이하) 양자화의 최대 적은 **outlier**입니다. 이 노트북은 outlier에 맞서는 처방들을
PyTorch로 하나씩 시뮬레이션합니다 — NVIDIA Blackwell의 **micro-tensor scaling**(그룹 양자화)부터
최신 **Hadamard 회전**까지.

1. FP4(E2M1) 포맷 — 표현 가능한 값이 15개뿐인 세계
2. Granularity 3단계 구현: **per-tensor → per-channel → per-group(32)**
3. Outlier 주입 실험 — per-tensor가 무너지는 이유 (MIT 6.5940 Lec06 p.14 재현)
4. **MXFP4 스타일**: 2의 거듭제곱(E8M0) 공유 scale
5. MNIST MLP 정확도로 granularity 효과 비교
6. 메모리 오버헤드 — effective bit width 계산 (4 + 8/32 = 4.25 bits)
7. **Activation 양자화** — KL divergence로 clipping 지점 찾기 (TensorRT 방식)
8. **AdaRound** — "올릴까 내릴까"를 학습하는 반올림
9. **Hadamard 회전** — 직교 회전으로 outlier를 아예 펴발라 버리기 (QuaRot/SpinQuant 계열)

Colab에서는 위에서부터 순서대로 실행하면 됩니다. (런타임 → 모두 실행)


## 0. 준비

In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. MNIST를 MLP로 학습 (짧게)

이전 실습과 같은 3층 MLP(784 → 256 → 128 → 10)를 2 에폭만 학습합니다. 목적은 **양자화해 볼 가중치**를 얻는 것입니다.

In [ ]:
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.1307,), (0.3081,))])
train_ds = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=transform)
test_ds  = torchvision.datasets.MNIST("./data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=512)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
    def forward(self, x):
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

@torch.no_grad()
def evaluate(model):
    model.eval()
    correct = 0
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
    return correct / len(test_ds)

model = MLP().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(2):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        opt.step()
    print(f"epoch {epoch+1}  loss {loss.item():.4f}  test acc {evaluate(model):.4f}")

acc_fp32 = evaluate(model)
print(f"\nFP32 baseline accuracy: {acc_fp32:.4f}")

## 2. FP4(E2M1) 포맷 — 표현 가능한 값이 15개

Blackwell의 FP4는 **E2M1** 포맷입니다: 부호 1비트 + 지수 2비트 + 가수 1비트.
표현 가능한 값은 딱 15개뿐입니다:

$$\pm\{0.5,\ 1,\ 1.5,\ 2,\ 3,\ 4,\ 6\} \ \cup\ \{0\}$$

INT4(균등 격자)와 달리 FP4의 격자는 **0 근처는 촘촘하고(간격 0.5) 끝으로 갈수록 성깁니다(4→6은 간격 2)**.
신경망 가중치는 0 근처에 종 모양으로 몰려 있으므로, 이 비균등 격자가 분포와 더 잘 맞습니다.

표현 범위가 [−6, 6]으로 극도로 좁기 때문에, 실수 텐서를 FP4로 넣으려면 **scale factor로 [−6, 6] 안에 밀어 넣어야** 합니다.
"scale을 얼마나 세밀한 단위로 둘 것인가"가 바로 granularity 문제입니다.

In [ ]:
# FP4 (E2M1) 양수 격자: 0, 0.5, 1, 1.5, 2, 3, 4, 6
FP4_POS = torch.tensor([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0])
FP4_GRID = torch.cat([-FP4_POS.flip(0)[:-1], FP4_POS])          # 15개 값
INT4_GRID = torch.arange(-7, 8, dtype=torch.float32) * (6.0 / 7)  # 같은 범위로 스케일한 INT4 (15개)

print("FP4  grid:", FP4_GRID.tolist())
print("INT4 grid:", [round(v, 2) for v in INT4_GRID.tolist()])

fig, ax = plt.subplots(figsize=(9, 1.8))
ax.scatter(FP4_GRID,  np.ones(15),  s=40, label="FP4 (E2M1) - denser near zero")
ax.scatter(INT4_GRID, np.zeros(15), s=40, marker="s", label="INT4 - uniform spacing")
ax.set_yticks([0, 1]); ax.set_yticklabels(["INT4", "FP4"]); ax.set_ylim(-0.6, 1.6)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=2)
ax.set_title("FP4 vs INT4 grid (same range [-6, 6])")
plt.tight_layout(); plt.show()

## 3. Granularity별 fake-quantization 구현

가중치 `W`를 그대로 두고 "FP4로 갔다 왔을 때의 값"으로 바꿔치기하는 **fake quantization**을 구현합니다.

공통 절차 (symmetric linear quantization):
1. 단위(텐서 전체 / 채널 / 그룹)별로 `amax = |W|max` 계산
2. `scale = amax / 6`  (6 = FP4 최대값)
3. `W / scale`을 FP4 격자에서 가장 가까운 값으로 스냅 → 다시 `× scale`

granularity 세 가지:

| granularity | scale 개수 (fc1 기준) | 비고 |
|---|---|---|
| `tensor` | 1 | outlier 하나가 전체 해상도를 지배 |
| `channel` | 256 (output channel당 1) | Lec06 p.14의 해법 |
| `group` | 6,272 (원소 32개당 1) | **micro-tensor scaling** (Blackwell, MXFP4의 블록 크기 = 32) |

그룹은 행 우선(row-major)으로 펼친 뒤 32개씩 자릅니다. Linear 가중치 `(out, in)`에서 행 하나가 input 차원을 따라 이어지므로, **그룹은 input 방향의 연속 블록** — 하드웨어가 내적(dot product)을 블록 단위로 처리하는 방향과 같습니다.

In [ ]:
def snap_to_grid(x, grid):
    """x의 각 원소를 grid에서 가장 가까운 값으로 스냅."""
    grid = grid.to(x.device)
    idx = (x.unsqueeze(-1) - grid).abs().argmin(dim=-1)
    return grid[idx]

def fake_quant(w, granularity="group", group_size=32, grid=FP4_GRID, pow2_scale=False):
    qmax = grid.abs().max().item()          # FP4면 6.0
    if granularity == "tensor":
        blocks = w.reshape(1, -1)
    elif granularity == "channel":
        blocks = w.reshape(w.shape[0], -1)  # output channel(행)마다 한 블록
    elif granularity == "group":
        assert w.numel() % group_size == 0
        blocks = w.reshape(-1, group_size)  # 원소 group_size개마다 한 블록
    else:
        raise ValueError(granularity)

    amax = blocks.abs().amax(dim=1, keepdim=True).clamp(min=1e-12)
    scale = amax / qmax
    if pow2_scale:                          # MXFP4 스타일: 2^k scale (섹션 5)
        scale = 2.0 ** torch.ceil(torch.log2(scale))
    wq = snap_to_grid(blocks / scale, grid) * scale
    return wq.reshape(w.shape)

# 층별 양자화 오차(MSE) 비교
print(f"{'layer':6s} {'per-tensor':>12s} {'per-channel':>12s} {'per-group(32)':>14s}")
for name, layer in [("fc1", model.fc1), ("fc2", model.fc2), ("fc3", model.fc3)]:
    w = layer.weight.data
    mses = [F.mse_loss(fake_quant(w, g), w).item() for g in ("tensor", "channel", "group")]
    print(f"{name:6s} {mses[0]:12.3e} {mses[1]:12.3e} {mses[2]:14.3e}")

세밀해질수록(오른쪽으로 갈수록) MSE가 줄어드는 것을 확인할 수 있습니다.
그런데 정상적으로 학습된 가중치에서는 차이가 몇 배 수준입니다. granularity가 진짜 극적으로 갈리는 순간은 **outlier가 있을 때**입니다.

## 4. Outlier 주입 실험 — per-tensor가 무너지는 이유

Lec06 p.14의 boxplot(MobileNetV2 depthwise layer의 채널 간 range 격차 100배)을 인위적으로 재현합니다.
fc1 가중치 복사본의 **원소 딱 하나**를 기존 최대값의 50배로 키운 뒤, granularity별 MSE가 어떻게 변하는지 봅니다.

In [ ]:
w = model.fc1.weight.data
w_out = w.clone()
w_out[0, 0] = w.abs().max() * 50          # 원소 1개(20만 개 중)만 outlier로

print(f"|W|max: {w.abs().max():.3f} -> {w_out.abs().max():.3f}\n")
print(f"{'granularity':14s} {'MSE (정상)':>12s} {'MSE (outlier)':>14s} {'증가율':>8s}")
mse_clean, mse_out = {}, {}
for g in ("tensor", "channel", "group"):
    mse_clean[g] = F.mse_loss(fake_quant(w, g), w).item()
    # outlier 원소 자체의 오차는 빼고, '나머지 가중치들'이 입은 피해만 측정
    diff = (fake_quant(w_out, g) - w_out).flatten()[1:]
    mse_out[g] = diff.pow(2).mean().item()
    print(f"{g:14s} {mse_clean[g]:12.3e} {mse_out[g]:14.3e} {mse_out[g]/mse_clean[g]:7.0f}x")

xs = np.arange(3)
plt.figure(figsize=(7, 3.5))
plt.bar(xs - 0.2, [mse_clean[g] for g in ("tensor", "channel", "group")], 0.4, label="clean weights")
plt.bar(xs + 0.2, [mse_out[g]   for g in ("tensor", "channel", "group")], 0.4, label="1 outlier injected")
plt.xticks(xs, ["per-tensor", "per-channel", "per-group(32)"])
plt.yscale("log"); plt.ylabel("MSE (log)"); plt.title("Damage radius of a single outlier")
plt.legend(); plt.tight_layout(); plt.show()

**결과 해석**: outlier 원소 하나가 `amax`를 50배로 키우면 scale도 50배가 됩니다.

- **per-tensor**: 0이 아닌 가장 작은 FP4 레벨이 `0.5 × scale ≈ 1.26`까지 올라가는데, 정상 가중치는 전부 `|w| ≤ 0.3` → **20만 개 가중치가 거의 전부 0으로 스냅**됩니다. MSE가 가중치 분산 수준(= 모든 정보를 잃었을 때의 값)까지 치솟음 — 층이 통째로 지워진 것
- **per-channel**: 피해가 해당 행(784개)에 갇힘 — 나머지 255개 채널은 무사
- **per-group(32)**: 피해가 같은 그룹 **31개**에만 갇힘 — 나머지 6,271개 그룹은 무사

이것이 micro-tensor scaling의 존재 이유입니다. 블록이 작을수록 outlier의 "폭발 반경"이 줄어듭니다.

## 5. MXFP4 스타일 — 2의 거듭제곱(E8M0) scale

실제 하드웨어 포맷(OCP MX 표준, Blackwell의 MXFP4)은 그룹(32개)마다 **FP32 scale이 아니라 8비트 지수(E8M0), 즉 2^k 형태의 scale**을 붙입니다.

- **저장**: scale이 8비트면 충분 (지수만 저장)
- **연산**: `× 2^k`는 곱셈기가 아니라 **지수 덧셈**으로 처리 — 하드웨어가 훨씬 싸다

대가는 scale이 2배 단위로만 움직인다는 것. 최적 scale(`amax/6`)을 2의 거듭제곱으로 올림(ceil)하면 clipping은 없지만 최대 2배 성긴 격자를 쓰게 됩니다. 오차가 얼마나 손해인지 확인해 봅니다.

> 참고: OCP MX 스펙은 실제로 `2^(floor(log2(amax)) − 2)`로 내림 + saturating cast를 쓰지만, 여기서는 clipping이 없는 올림 방식으로 단순화했습니다.

In [ ]:
print(f"{'layer':6s} {'group32 + FP scale':>19s} {'group32 + 2^k scale':>20s} {'손해':>7s}")
for name, layer in [("fc1", model.fc1), ("fc2", model.fc2), ("fc3", model.fc3)]:
    w = layer.weight.data
    mse_fp   = F.mse_loss(fake_quant(w, "group"), w).item()
    mse_pow2 = F.mse_loss(fake_quant(w, "group", pow2_scale=True), w).item()
    print(f"{name:6s} {mse_fp:19.3e} {mse_pow2:20.3e} {mse_pow2/mse_fp:6.2f}x")

2^k scale의 MSE 손해는 1.0~1.4배 수준 — **8비트 저장 + 지수 덧셈이라는 하드웨어 이득에 비하면 싼 대가**입니다.
outlier 상황에서 per-tensor → per-group이 만드는 차이(층 전멸 vs 그룹 31개 피해)와는 비교가 안 되게 작은 손해입니다.

## 6. 모델 정확도 비교

이제 실제로 MLP의 모든 Linear 가중치를 각 방식으로 양자화하고 MNIST 정확도를 비교합니다.

In [ ]:
@torch.no_grad()
def eval_quantized(**kwargs):
    m = copy.deepcopy(model)
    for mod in m.modules():
        if isinstance(mod, nn.Linear):
            mod.weight.copy_(fake_quant(mod.weight.data, **kwargs))
    return evaluate(m)

configs = [
    ("FP32 baseline",        None),
    ("FP4 per-tensor",       dict(granularity="tensor")),
    ("FP4 per-channel",      dict(granularity="channel")),
    ("FP4 per-group(32)",    dict(granularity="group")),
    ("MXFP4 (group32, 2^k)", dict(granularity="group", pow2_scale=True)),
    ("INT4 per-group(32)",   dict(granularity="group", grid=INT4_GRID)),
]

names, accs = [], []
for name, kw in configs:
    acc = acc_fp32 if kw is None else eval_quantized(**kw)
    names.append(name); accs.append(acc)
    print(f"{name:22s} acc = {acc:.4f}  (baseline 대비 {acc - acc_fp32:+.4f})")

plt.figure(figsize=(8, 3.5))
bars = plt.barh(names[::-1], accs[::-1])
bars[-1].set_color("gray")
plt.xlim(min(accs) - 0.01, max(accs) + 0.005)
plt.xlabel("MNIST test accuracy"); plt.title("4-bit weight quantization accuracy")
plt.tight_layout(); plt.show()

> 작은 MLP + MNIST는 워낙 쉬운 문제라 차이가 크지 않을 수 있습니다. 그래도 per-tensor ≤ per-channel ≤ per-group 경향과,
> MXFP4(2^k scale)가 FP scale과 거의 같은 정확도를 내는 것은 확인됩니다. 태스크가 어렵고 모델이 민감할수록(강의의 MobileNetV2처럼) 격차가 벌어집니다.
>
> 또 하나의 관찰: **그룹이 32개로 잘게 쪼개지면 INT4 균등 격자도 FP4만큼 잘 동작**합니다. 그룹 안에서는 분포의 꼬리가 짧아지기 때문입니다.
> 비균등(FP) 격자의 이점은 블록이 클수록 — 즉 한 scale이 감당해야 할 분포가 넓고 꼬리가 길수록 — 커집니다.

## 7. 메모리 오버헤드 — effective bit width

scale을 세밀하게 둘수록 scale 자체의 저장 비용이 붙습니다. 원소당 실효 비트폭으로 환산하면:

$$\text{effective bits} = \text{element bits} + \frac{\text{scale bits}}{\text{group size}}$$

In [ ]:
schemes = [
    # (이름, 원소 비트, scale 비트, 그룹 크기)
    ("FP4 per-tensor",              4,  32, 200704),   # fc1 기준 — scale 1개는 사실상 공짜
    ("FP4 group32 + FP16 scale",    4,  16, 32),
    ("MXFP4 (group32 + E8M0)",      4,   8, 32),
    ("VS-Quant (INT4 scale/16개)",  4,   4, 16),       # + coarse FP scale은 텐서당 1개라 무시
    ("NVFP4 (group16 + FP8 scale)", 4,   8, 16),
]
print(f"{'scheme':30s} {'effective bits':>14s} {'FP32 대비 압축률':>16s}")
for name, eb, sb, gs in schemes:
    bits = eb + sb / gs
    print(f"{name:30s} {bits:14.3f} {32 / bits:15.1f}x")

## 7. Activation 양자화 — KL divergence로 어디서 자를지 찾기

지금까지는 전부 **weight** 얘기였습니다. weight는 정적이라 min/max를 정확히 알 수 있지만,
**activation은 입력마다 range가 바뀝니다.** 그래서 calibration 배치를 흘려 통계를 모으고,
"어디서 자를(clip) 것인가"를 정해야 합니다.

특히 ReLU 출력은 꼬리가 아주 길어서, 관측된 **max로 scale을 잡으면 낭비가 큽니다** —
극소수 outlier 때문에 대다수 값의 해상도가 뭉개집니다. TensorRT의 해법은 **KL divergence 최소화**:

1. calibration으로 activation 히스토그램 수집 (여기서는 2048 bins)
2. 후보 clip 지점 T를 순회하며:
   - **P** = T에서 자른 원본 히스토그램 (T 밖 outlier는 마지막 bin에 합산 = saturate)
   - **Q** = P를 n_levels개 레벨로 양자화했다가 다시 펼친 히스토그램
3. D_KL(P‖Q)가 **최소**인 T 선택 — "이 지점에서 자를 때 정보 손실이 가장 작다"

분포 형태에 대한 가정(Gaussian/Laplace)이 전혀 없는 **비파라메트릭** 방법입니다.

In [ ]:
# fc1, fc2의 ReLU 출력(activation)을 calibration 배치 20개로 수집
acts1, acts2 = [], []
with torch.no_grad():
    for i, (x, _) in enumerate(train_loader):
        if i >= 20: break
        a1 = F.relu(model.fc1(x.to(device).flatten(1)))
        a2 = F.relu(model.fc2(a1))
        acts1.append(a1.cpu()); acts2.append(a2.cpu())
acts1 = torch.cat(acts1).flatten().numpy()
acts2 = torch.cat(acts2).flatten().numpy()
print(f"fc1 act: max {acts1.max():.2f}, 99.9% 분위수 {np.percentile(acts1, 99.9):.2f}")
print(f"fc2 act: max {acts2.max():.2f}, 99.9% 분위수 {np.percentile(acts2, 99.9):.2f}")

N_BINS = 2048

def kl_divergence(p, q):
    p = p / p.sum(); q = q / q.sum()
    mask = p > 0
    return float((p[mask] * np.log(p[mask] / np.maximum(q[mask], 1e-12))).sum())

def kl_threshold(acts, n_levels, n_bins=N_BINS, stride=16):
    """D_KL(P||Q)가 최소가 되는 clip 지점 T와 (후보, KL) 곡선을 반환.

    P는 outlier를 마지막 bin에 saturate한 분포, Q는 [0,T) '안쪽만' n_levels로
    양자화했다 펼친 분포. 둘의 차이가 clipping 손실(잘린 질량) + rounding 손실
    (병합으로 뭉개진 해상도)을 동시에 잡는다."""
    pos = acts[acts > 0]                            # ReLU의 0은 오차 없이 표현되므로 제외
    hist, edges = np.histogram(pos, bins=n_bins, range=(0, pos.max()))
    cands, kls = [], []
    for i in range(n_levels * 8, n_bins + 1, stride):
        P = hist[:i].astype(np.float64).copy()
        P[-1] += hist[i:].sum()                     # 잘린 outlier를 마지막 bin에 saturate
        Q = np.zeros(i)                             # 원본 hist[:i]를 n_levels로 병합했다 펼침
        idx = np.linspace(0, i, n_levels + 1).astype(int)
        for j in range(n_levels):
            seg = hist[idx[j]:idx[j+1]].astype(np.float64)   # saturate 전 분포에서 구성
            nz = seg > 0
            if nz.any():
                Q[idx[j]:idx[j+1]][nz] = seg[nz].sum() / nz.sum()
        cands.append(edges[i]); kls.append(kl_divergence(P, Q))
    return cands[int(np.argmin(kls))], (cands, kls)

T1, curve1 = kl_threshold(acts1, n_levels=16)   # 4-bit(16 levels) 기준
T2, _      = kl_threshold(acts2, n_levels=16)
print(f"\nKL-optimal clip: fc1 T={T1:.2f} (max {acts1.max():.2f}), fc2 T={T2:.2f} (max {acts2.max():.2f})")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(acts1[acts1 > 0], bins=300, log=True, alpha=0.7)
axes[0].axvline(acts1.max(), color="gray", ls="--", label=f"max = {acts1.max():.1f}")
axes[0].axvline(T1, color="red", ls="-", label=f"KL-optimal T = {T1:.1f}")
axes[0].set_title("fc1 ReLU activations (log scale)"); axes[0].set_xlabel("activation value")
axes[0].legend()
axes[1].plot(*curve1)
axes[1].axvline(T1, color="red", ls="-")
axes[1].set_title("KL divergence vs clip threshold T"); axes[1].set_xlabel("T"); axes[1].set_ylabel("KL(P||Q)")
plt.tight_layout(); plt.show()

KL 곡선이 **U자형**인 것이 핵심입니다: T가 너무 작으면 saturation 손실(잘림), 너무 크면 rounding 손실(격자가 성겨짐).
KL이 그 균형점을 찾아줍니다. 이제 실제 정확도로 확인합니다 — **activation을 max로 자를 때 vs KL 지점에서 자를 때**.

In [ ]:
@torch.no_grad()
def eval_act_quant(clips, n_bits):
    """weight는 FP32 그대로 두고 ReLU activation만 n_bits로 fake-quant해서 평가."""
    qmax = 2 ** n_bits - 1                    # ReLU 출력은 0 이상이므로 unsigned
    def q(a, c):
        s = c / qmax
        return (a.clamp(0, c) / s).round() * s
    correct = 0
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        a1 = q(F.relu(model.fc1(x.flatten(1))), clips[0])
        a2 = q(F.relu(model.fc2(a1)), clips[1])
        correct += (model.fc3(a2).argmax(1) == y).sum().item()
    return correct / len(test_ds)

clip_max = (float(acts1.max()), float(acts2.max()))
clip_kl  = (T1, T2)
print(f"{'bits':>5s} {'clip=max':>10s} {'clip=KL':>10s}   (FP32 baseline {acc_fp32:.4f})")
for b in (4, 3, 2):
    print(f"{b:5d} {eval_act_quant(clip_max, b):10.4f} {eval_act_quant(clip_kl, b):10.4f}")

비트가 낮아질수록 **KL clipping의 이득이 커지는** 것을 확인할 수 있습니다. outlier 몇 개를 포기하고
대다수 값의 해상도를 지키는 쪽이 남는 장사라는 것 — weight의 granularity 때와 같은 교훈이 clipping에서 반복됩니다.

## 8. AdaRound — "올릴까 내릴까"를 학습한다

scale과 clip을 다 정한 뒤에도 자유도가 하나 남습니다: **반올림**. 가장 가까운 격자점으로 보내는
rounding-to-nearest(RTN)는 가중치 **하나하나의** 오차를 최소화하지만, 같은 층의 가중치들은 같은 입력에
곱해져 하나의 출력을 만들므로 **개별 최선의 합 ≠ 전체 최선**입니다.

AdaRound(Nagel et al., 2020)는 기준을 바꿉니다 — 가중치가 아니라 **층 출력을 가장 잘 복원하는 반올림**:

$$\arg\min_{\mathbf{V}} \|\mathbf{W}\mathbf{x} - \lfloor\lfloor\mathbf{W}\rfloor + h(\mathbf{V})\rceil\,\mathbf{x}\|_F^2 + \lambda f_{reg}(\mathbf{V})$$

- 각 가중치를 `floor`(내림) 또는 `floor+1`(올림) 둘 중 하나로 — 그 선택 h(V)∈(0,1)를 gradient로 학습
- **h**: rectified sigmoid (연속 완화 → gradient가 흐름)
- **f_reg**: h를 0 또는 1로 밀어붙이는 정규화 (β를 크게→작게 annealing)
- calibration 데이터 몇 배치, 층별 독립 최적화 — 라벨도 전체 재학습도 불필요 = "(almost) PTQ"

In [ ]:
def rtn_quant(W, n_bits):
    """round-to-nearest, per-channel symmetric INT."""
    qmax = 2 ** (n_bits - 1) - 1
    s = W.abs().amax(dim=1, keepdim=True).clamp(min=1e-12) / qmax
    return ((W / s).round().clamp(-qmax - 1, qmax)) * s

def adaround_quant(W, X, n_bits, iters=800, lam=0.01):
    """AdaRound: 층 입력 X로 출력 재구성 오차를 최소화하는 올림/내림을 학습."""
    qmax = 2 ** (n_bits - 1) - 1
    s = W.abs().amax(dim=1, keepdim=True).clamp(min=1e-12) / qmax
    Wf = torch.floor(W / s)
    frac = (W / s - Wf).clamp(0.01, 0.99)
    V = (-torch.log(1.2 / (frac + 0.1) - 1)).clone().requires_grad_(True)  # h(V)=frac에서 시작
    opt = torch.optim.Adam([V], lr=1e-2)
    Y = X @ W.T
    for it in range(iters):
        h = (torch.sigmoid(V) * 1.2 - 0.1).clamp(0, 1)                     # rectified sigmoid
        Wq = (Wf + h).clamp(-qmax - 1, qmax) * s
        beta = 20 - 18 * it / iters                                        # 20 → 2 annealing
        f_reg = (1 - (2 * h - 1).abs().pow(beta)).sum()
        loss = F.mse_loss(X @ Wq.T, Y) + lam * f_reg
        opt.zero_grad(); loss.backward(); opt.step()
    h_bin = ((torch.sigmoid(V) * 1.2 - 0.1) > 0.5).float()
    flipped = (h_bin != (frac > 0.5).float()).float().mean().item()        # RTN과 다른 결정 비율
    return ((Wf + h_bin).clamp(-qmax - 1, qmax) * s).detach(), flipped

# calibration 입력 (라벨 불필요, 1024장)
x_cal = torch.cat([x for x, _ in list(train_loader)[:8]])[:1024].to(device).flatten(1)
with torch.no_grad():
    X1 = x_cal
    X2 = F.relu(model.fc1(X1))
    X3 = F.relu(model.fc2(X2))

@torch.no_grad()
def eval_with_weights(w1, w2, w3):
    m = copy.deepcopy(model)
    m.fc1.weight.copy_(w1); m.fc2.weight.copy_(w2); m.fc3.weight.copy_(w3)
    return evaluate(m)

results_round = {}
for b in (3, 2):
    acc_rtn = eval_with_weights(*[rtn_quant(l.weight.data, b) for l in (model.fc1, model.fc2, model.fc3)])
    ada = [adaround_quant(l.weight.data, X, b) for l, X in
           [(model.fc1, X1), (model.fc2, X2), (model.fc3, X3)]]
    acc_ada = eval_with_weights(*[w for w, _ in ada])
    flip = np.mean([f for _, f in ada])
    results_round[b] = (acc_rtn, acc_ada)
    print(f"INT{b}: RTN {acc_rtn:.4f} -> AdaRound {acc_ada:.4f}  (반올림 결정이 바뀐 비율 {flip:.1%})")

xs = np.arange(2)
plt.figure(figsize=(6, 3.2))
plt.bar(xs - 0.18, [results_round[b][0] for b in (3, 2)], 0.36, label="RTN (round-to-nearest)")
plt.bar(xs + 0.18, [results_round[b][1] for b in (3, 2)], 0.36, label="AdaRound")
plt.xticks(xs, ["INT3", "INT2"]); plt.ylabel("MNIST test accuracy")
plt.title("Learned rounding vs round-to-nearest"); plt.legend()
plt.tight_layout(); plt.show()

반올림 결정의 일부만 바뀌었는데도 저비트에서 정확도가 눈에 띄게 회복됩니다. "가장 가까운 값"이라는
당연해 보이는 선택이 사실 최적이 아니었다는 것이 AdaRound의 교훈입니다.

## 9. Hadamard 회전 — outlier를 아예 펴발라 버리기

지금까지의 처방(그룹 scale, clipping, 학습된 반올림)은 전부 **outlier와의 공존**이었습니다.
최신 LLM 양자화(QuaRot, SpinQuant 등)는 더 급진적입니다 — **outlier가 없는 좌표계로 회전해 버리자.**

핵심은 **계산 불변성(computational invariance)**: 직교 행렬 R (RRᵀ = I)에 대해

$$\mathbf{y} = \mathbf{W}\mathbf{x} = (\mathbf{W}\mathbf{R}^\top)(\mathbf{R}\mathbf{x})$$

가중치를 WRᵀ로, 입력을 Rx로 미리 바꿔도 **출력은 수학적으로 동일**합니다. 그런데 R로 **Hadamard 행렬**
(±1 성분의 직교 행렬, H/√n)을 쓰면 회전 후 각 성분은 원래 성분들의 ±평균이 되어 —
**한 자리에 몰려 있던 outlier가 모든 차원에 1/√n 크기로 얇게 퍼집니다.** 분포가 가우시안에 가까워지고
(첨도 kurtosis ↓), outlier가 사라진 텐서는 저비트 양자화가 쉬워집니다.

fc2(입력 256차원 — 2의 거듭제곱이라 Hadamard가 바로 존재)에 outlier를 주입하고 실험합니다.

In [ ]:
def hadamard_matrix(n):
    """Sylvester 구성법: H_2n = [[H, H], [H, -H]]. n은 2의 거듭제곱."""
    H = np.array([[1.0]])
    while H.shape[0] < n:
        H = np.block([[H, H], [H, -H]])
    return H / np.sqrt(n)

Hd = torch.tensor(hadamard_matrix(256), dtype=torch.float32, device=device)
print("직교성 검증 |HH^T - I|_max =", (Hd @ Hd.T - torch.eye(256, device=device)).abs().max().item())

def kurtosis(t):
    t = t.flatten().double()
    return (((t - t.mean()) ** 4).mean() / t.var() ** 2).item()

# fc2 가중치에 outlier 주입 (섹션 4와 동일한 시나리오)
W = model.fc2.weight.data.clone()
W[0, 0] = W.abs().max() * 50
W_rot = W @ Hd.T                                   # 회전된 가중치

print(f"\n|W|max:      원본 {W.abs().max():.2f}  ->  회전 후 {W_rot.abs().max():.2f}")
print(f"kurtosis:    원본 {kurtosis(W):8.1f}  ->  회전 후 {kurtosis(W_rot):6.1f}  (가우시안 = 3)")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
axes[0].hist(W.flatten().cpu().numpy(), bins=200, log=True)
axes[0].set_title(f"before rotation  (kurtosis {kurtosis(W):.0f})")
axes[1].hist(W_rot.flatten().cpu().numpy(), bins=200, log=True)
axes[1].set_title(f"after Hadamard rotation  (kurtosis {kurtosis(W_rot):.1f})")
for ax in axes: ax.set_xlabel("weight value")
plt.suptitle("Hadamard rotation smears the outlier across all dimensions")
plt.tight_layout(); plt.show()

In [ ]:
# 출력 오차로 확인: y = X W^T (X는 fc2의 실제 입력 = relu(fc1) activation)
X = X2[:1024]                                       # (1024, 256) calibration activations
X_rot = X @ Hd.T                                    # 입력도 같은 회전: y = (X H^T)(W H^T)^T

def rel_err(y, ref): return ((y - ref).norm() / ref.norm()).item()

# 회전 불변성 검증 (양자화 없이): 오차가 부동소수점 수준이어야 함
print("불변성 검증 rel_err =", rel_err(X_rot @ W_rot.T, X @ W.T))

# 출력(128차원)쪽 회전도 추가 — QuaRot처럼 양방향: W' = H_out W H_in^T
# 출력이 H_out y로 회전되지만, 실제 네트워크에서는 다음 층에 접어 넣으면 된다.
Hd_out = torch.tensor(hadamard_matrix(128), dtype=torch.float32, device=device)
W_rot2 = Hd_out @ W @ Hd.T
print(f"|W|max: 한쪽 회전 {W_rot.abs().max():.2f} -> 양쪽 회전 {W_rot2.abs().max():.2f}"
      f"  (kurtosis {kurtosis(W_rot2):.1f})")

W_clean = model.fc2.weight.data                     # outlier 없는 원본 (참조선)
rows = [("outlier 없음 (참조)",       W_clean, X),
        ("outlier + 회전 없음",       W,       X),
        ("outlier + H 입력쪽만",      W_rot,   X_rot),
        ("outlier + H 양쪽 (QuaRot)", W_rot2,  X_rot)]
print(f"\nFP4 weight 양자화 후 출력 상대오차:")
print(f"{'':26s} {'per-tensor':>12s} {'per-group(32)':>14s}")
for name, w, x in rows:
    ref = x @ w.T
    errs = [rel_err(x @ fake_quant(w, g).T, ref) for g in ("tensor", "group")]
    print(f"{name:26s} {errs[0]:12.4f} {errs[1]:14.4f}")

읽는 법:

- **입력쪽 회전만**: outlier가 자기 **행 안**에서 256등분되어 퍼짐 — 오차가 크게 줄지만, 그 행 전체가
  커진 채 남아 참조 수준까지는 못 감
- **양쪽 회전(QuaRot 방식)**: 출력 방향으로도 퍼져 outlier의 흔적이 사실상 소멸 (kurtosis ≈ 3, |W|max가
  outlier 없던 원본 수준) — **per-tensor 오차가 참조 수준에 근접**
- **per-group은 회전 이득이 없음**: 그룹 scale이 이미 outlier를 가두고 있었기 때문. 즉 **회전의 이득은
  scale이 성길수록(coarse) 크다** — 그래서 회전은 그룹 scale의 대체재이자, scale을 촘촘히 붙이기 어려운 곳
  (W4A4의 activation 등)의 강력한 보완재
- 여기는 256차원이라 outlier가 1/√256 = 1/16로 줄지만, 실제 LLM(4096차원 이상)에서는 1/64 이하로 —
  차원이 클수록 회전의 효과는 더 완벽해짐

그룹 scale(저장 오버헤드 +0.25bit)과 달리 회전은 **추가 저장이 0**입니다 — WRᵀ는 오프라인에서 미리 계산해 두면 되고,
Rx는 Hadamard 변환의 구조 덕분에 O(n log n)으로 온라인 처리됩니다.

> 실제 QuaRot/SpinQuant는 회전을 인접 층에 접어 넣고(fold), residual/attention 경로까지 불변성을 유지하며
> 회전합니다. LLM의 W4A4 양자화가 실용화된 핵심 트릭입니다. SpinQuant는 여기서 한 발 더 나아가
> 회전 행렬 자체를 학습합니다.

## 정리

저비트 양자화의 적은 outlier 하나였고, 처방은 네 단계로 진화했습니다.

| 처방 | 아이디어 | 비용 |
|---|---|---|
| **그룹 scale** (micro-tensor scaling) | outlier의 피해를 그룹 31개 안에 가둔다 | scale 저장 (+0.25 bit) |
| **KL clipping** | activation의 outlier 몇 개를 포기하고 해상도를 지킨다 | calibration 탐색 |
| **AdaRound** | 반올림을 층 출력 기준으로 학습한다 | 층별 짧은 최적화 |
| **Hadamard 회전** | outlier가 없는 좌표계로 회전한다 (분포 자체를 교정) | 추가 저장 0, 온라인 O(n log n) |

- **per-tensor → per-channel → per-group(32)**: FP4는 표현값 15개뿐이라 scale의 세밀함이 생사를 가른다
- **MXFP4**: 그룹(32개)마다 2^k(E8M0) scale — 정확도 손해 미미, 하드웨어는 지수 덧셈만. effective 4.25 bits. Blackwell이 네이티브 지원
- **KL clipping**: 비트가 낮을수록 "max로 자르기"와의 격차가 벌어진다
- **AdaRound**: 반올림 결정 일부만 바뀌어도 저비트 정확도가 회복 — "가장 가까운 값"은 최적이 아니다
- **Hadamard 회전**: 계산 불변성 y = (WRᵀ)(Rx) 덕분에 공짜로 outlier를 제거 — per-tensor scale마저 살려낸다

## 참고자료

- MIT 6.5940 TinyML, Lecture 6: Quantization Part II (Quantization Granularity / Dynamic Range Clipping / Rounding)
- OCP Microscaling Formats (MX) Specification v1.0
- *VS-Quant: Per-Vector Scaled Quantization for Accurate Low-Precision Neural Network Inference* (Dai et al., MLSys 2021)
- *8-bit Inference with TensorRT* (Szymon Migacz, 2017) — KL divergence calibration
- *Up or Down? Adaptive Rounding for Post-Training Quantization* (Nagel et al., PMLR 2020) — AdaRound
- *QuaRot: Outlier-Free 4-Bit Inference in Rotated LLMs* (Ashkboos et al., NeurIPS 2024)
- *SpinQuant: LLM Quantization with Learned Rotations* (Liu et al., 2024)
- NVIDIA, *Blackwell Architecture for Generative AI*